In [1]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

In [4]:
# pobranie i załadowanie danych

mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples, tf.int64)

In [14]:
# normalizacja danych

def scale(image, label):
  image = tf.cast(image, tf.float32)
  image /= 255.
  return image, label

scaled_train_and_validation_dataa = mnist_train.map(scale)

test_data = mnist_test.map(scale)

In [15]:
BUFFER_SIZE = 1000

shuffled_train_and_validation_data = scaled_train_and_validation_dataa.shuffle(BUFFER_SIZE)

validation_data = shuffled_train_and_validation_data.take(num_validation_samples)

train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

In [16]:
BATCH_SIZE = 100

train_data = train_data.batch(BATCH_SIZE)

validation_data = validation_data.batch(num_validation_samples)

test_data = test_data.batch(num_test_samples)

validation_inputs,validation_targets = next(iter(validation_data))
print(validation_inputs.shape, validation_targets.shape)

(6000, 28, 28, 1) (6000,)


Budowa modelu

In [17]:
inputs_size = 784
output_size = 10

hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),

    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),

    tf.keras.layers.Dense(output_size, activation='softmax')

])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [18]:
NUM_EPOCHS = 30

early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)

model.fit(train_data,
          epochs=NUM_EPOCHS,
          callbacks=[early_stopping],
          validation_data=(validation_inputs, validation_targets),
          verbose = 1
          )

Epoch 1/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8889 - loss: 0.3990 - val_accuracy: 0.9392 - val_loss: 0.2113
Epoch 2/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9481 - loss: 0.1793 - val_accuracy: 0.9515 - val_loss: 0.1664
Epoch 3/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9603 - loss: 0.1353 - val_accuracy: 0.9547 - val_loss: 0.1486
Epoch 4/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9679 - loss: 0.1103 - val_accuracy: 0.9548 - val_loss: 0.1477
Epoch 5/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9731 - loss: 0.0934 - val_accuracy: 0.9613 - val_loss: 0.1274
Epoch 6/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9761 - loss: 0.0800 - val_accuracy: 0.9643 - val_loss: 0.1192
Epoch 7/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9801 - loss: 0.0690 - val_accuracy: 0.9665 - val_loss: 0.1169
Epoch 8/30
540/540 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9817 - loss: 0.0612 - val_accuracy

In [19]:
test_loss, test_accuracy = model.evaluate(test_data)
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step - accuracy: 0.9698 - loss: 0.1108
Test loss: 0.11. Test accuracy: 96.98%


Zadanie:
Wykorzystując bibliotekę  opencv-python  oraz metodę  model.predict()  załaduj do modelu pojedynczy obrazek cyfry pisanej odręcznie i sprawdź skuteczność modelu.

In [35]:
image = cv2.imread('obraz.jpg')
image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
image = cv2.resize(image, (28, 28))
image = image.reshape(1, 28, 28, 1)
image = image / 255.0
prediction = model.predict(image)
print('Prediction: ',np.argmax(prediction))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Prediction:  3
